# DICOM CT -> 3D reconstruction
Run the two cells in order. Cell 1 = setup (once per session). Cell 2 = the whole pipeline.

In [ ]:
!pip install pydicom -q          # plotly, scipy and scikit-image come preinstalled in Colab

from google.colab import drive
drive.mount('/content/drive')

# Unzip the zip you uploaded to Drive (edit the path to match yours)
!unzip -q "/content/drive/MyDrive/manifest-1791342733858.zip" -d /content/dicom

## Pipeline: load -> stack -> resample -> masks -> surfaces -> interactive 3D

In [ ]:
import numpy as np
import pydicom
from pathlib import Path
from scipy import ndimage
from skimage import measure
import plotly.graph_objects as go

# ---------- SETTINGS (change these) ----------
DICOM_DIR = "/content/dicom"   # folder that contains your .dcm files
ISO       = 2.0                # cube size in mm (smaller = more detail, slower)
BONE_HU   = 250                # voxels denser than this count as bone
AIR_HU    = -300               # voxels lighter than this count as air

# ---------- 1. LOAD ----------
files = sorted(Path(DICOM_DIR).rglob("*.dcm"))
slices = [pydicom.dcmread(f) for f in files]
print(len(slices), "slices, modality:", slices[0].Modality)
assert len({s.SeriesInstanceUID for s in slices}) == 1, "folder has more than one series"

# ---------- 2. STACK into a 3D block of Hounsfield units ----------
slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))      # feet -> head
vol = np.stack([s.pixel_array * float(s.RescaleSlope) + float(s.RescaleIntercept)
                for s in slices]).astype(np.float32)

dz = abs(float(slices[1].ImagePositionPatient[2]) - float(slices[0].ImagePositionPatient[2]))
dy, dx = map(float, slices[0].PixelSpacing)
print("block shape:", vol.shape, "| voxel size (mm):", dz, dy, dx)

# ---------- 3. RESAMPLE so every voxel is a cube ----------
vol_iso = ndimage.zoom(vol, (dz/ISO, dy/ISO, dx/ISO), order=1)
print("resampled shape:", vol_iso.shape)

# ---------- 4. MASKS (True/False blocks) ----------
not_air = ndimage.binary_opening(vol_iso > AIR_HU, iterations=2)     # drops thin table/specks
labels, n = ndimage.label(not_air)                                   # number each blob
sizes = ndimage.sum(not_air, labels, range(1, n + 1))
body = labels == (1 + int(np.argmax(sizes)))                         # keep the biggest blob
body = np.stack([ndimage.binary_fill_holes(sl) for sl in body])     # fill lungs/gas, one axial slice at a time

bone = (vol_iso > BONE_HU) & ndimage.binary_erosion(body, iterations=2)
labels, n = ndimage.label(bone)
sizes = ndimage.sum(bone, labels, range(1, n + 1))
bone = np.isin(labels, [i + 1 for i, s in enumerate(sizes) if s > 500])   # drop tiny specks
print("body voxels:", int(body.sum()), "| bone voxels:", int(bone.sum()))

# ---------- 5. SURFACES (marching cubes) ----------
def surface(mask, sigma, step):
    smooth = np.pad(ndimage.gaussian_filter(mask.astype(np.float32), sigma), 1)
    verts, faces, _, _ = measure.marching_cubes(smooth, level=0.5,
                                                spacing=(ISO,) * 3, step_size=step)
    return (verts - ISO).astype(np.float32), faces

skin_v, skin_f = surface(body, sigma=1.5, step=2)
bone_v, bone_f = surface(bone, sigma=0.8, step=1)
print("triangles -> skin:", len(skin_f), "| bone:", len(bone_f))

# ---------- 6. DRAW (interactive) ----------
def mesh(v, f, color, opacity, name):
    return go.Mesh3d(x=v[:, 2], y=v[:, 1], z=v[:, 0],           # our array is (z, y, x)
                     i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     color=color, opacity=opacity, name=name, showlegend=True,
                     lighting=dict(ambient=0.45, diffuse=0.8, specular=0.15))

fig = go.Figure([mesh(skin_v, skin_f, "tan", 0.12, "skin"),
                 mesh(bone_v, bone_f, "ivory", 1.0, "bone")])
fig.update_layout(
    scene=dict(aspectmode="data",
               xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
               camera=dict(eye=dict(x=0, y=-2.2, z=0.3))),       # view from the front
    height=700, margin=dict(l=0, r=0, t=0, b=0))
fig.show()
